### Задание для самостоятельной работы (ДЗ 6)

1. Запустите `DBSCAN` на датасете Wine (с параметрами `eps=1.5, min_samples=5`).
2. Посчитайте `silhouette_score` и `adjusted_rand_score`.
3. Сравните с KMeans. Какой алгоритм лучше справился с поиском истинных сортов? Почему?

In [1]:
import numpy as np
from sklearn.datasets import load_wine
from sklearn.cluster import DBSCAN, KMeans
from sklearn.metrics import silhouette_score, adjusted_rand_score
from sklearn.preprocessing import StandardScaler
import warnings

warnings.filterwarnings('ignore')

wine = load_wine()
X = wine.data
y = wine.target

print("Размер данных:", X.shape)
print("Количество классов:", len(np.unique(y)))

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

Размер данных: (178, 13)
Количество классов: 3


## 1. Запуск DBSCAN

In [2]:
dbscan = DBSCAN(eps=1.5, min_samples=5)
dbscan_labels = dbscan.fit_predict(X_scaled)

n_clusters = len(set(dbscan_labels) - {-1})
n_noise = list(dbscan_labels).count(-1)

print(f"Количество найденных кластеров: {n_clusters}")
print(f"Количество шумовых точек: {n_noise} из {len(X_scaled)}")
print(f"Уникальные метки: {np.unique(dbscan_labels)}")

Количество найденных кластеров: 0
Количество шумовых точек: 178 из 178
Уникальные метки: [-1]


## 2. Метрики для DBSCAN



In [3]:
if n_clusters > 1:
    silhouette_db = silhouette_score(X_scaled, dbscan_labels)
else:
    silhouette_db = "Не применимо (найдено меньше 2 кластеров)"

ari_db = adjusted_rand_score(y, dbscan_labels)

print(f"DBSCAN Silhouette Score: {silhouette_db}")
print(f"DBSCAN Adjusted Rand Score: {ari_db:.4f}")

DBSCAN Silhouette Score: Не применимо (найдено меньше 2 кластеров)
DBSCAN Adjusted Rand Score: 0.0000


## 3. Запуск KMeans и метрики

In [4]:
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
kmeans_labels = kmeans.fit_predict(X_scaled)

silhouette_km = silhouette_score(X_scaled, kmeans_labels)
ari_km = adjusted_rand_score(y, kmeans_labels)

print(f"KMeans Silhouette Score: {silhouette_km:.4f}")
print(f"KMeans Adjusted Rand Score: {ari_km:.4f}")

KMeans Silhouette Score: 0.2849
KMeans Adjusted Rand Score: 0.8975


## 4. Ответы на вопросы задания

**1. Запуск DBSCAN с параметрами (eps=1.5, min_samples=5):**
Был успешно выполнен. Результат показал, что алгоритм не выделил ни одного кластера и отнес все 178 объектов к шуму (метка -1).

**2. Метрики:**
- **DBSCAN:** Silhouette Score посчитать невозможно (так как нет кластеров), Adjusted Rand Score = 0.0 (так как алгоритм ничего не предсказал).
- **KMeans:** Silhouette Score ≈ 0.285, Adjusted Rand Score ≈ 0.897.

**3. Сравнение и объяснение (Почему?):**
**KMeans справился гораздо лучше.** ARI почти 0.9 означает, что KMeans практически идеально восстановил истинные сорта винограда.

**Почему DBSCAN провалился с заданными параметрами, а KMeans — нет:**
1. **Специфика DBSCAN:** Параметр `eps` задает радиус поиска соседей. Данные были стандартизированы, а признаки в датасете Wine имеют разную природу (много измерений). При `eps=1.5` расстояние между большинством точек в 13-мерном пространстве оказывается больше 1.5. DBSCAN просто не находит соседей и считает каждую точку выбросом.
2. **Специфика KMeans:** Мы *заранее знаем* количество классов (3) и передаем это в `n_clusters=3`. Алгоритм ищет центры масс и притягивает к ним точки. В данном случае вина действительно кластеризуются в три плотные группы (что подтверждается высоким ARI).

**Вывод:** Для этого датасета (компактные сферические кластеры) KMeans подходит лучше. DBSCAN хорош для данных сложной формы и с неизвестным числом кластеров, но требует тонкой настройки `eps` (например, через график k-distance).